# ASCENT-G Phase 0 Pilot: GSM8K × Qwen2.5-1.5B-Instruct

**Phase:** 0 — Pipeline validation only  
**Model:** `Qwen2.5-1.5B-Instruct` (registered primary)  
**Task:** GSM8K  
**Method:** GRPO via TRL  

**Acceptance criteria (from execution-plan.md):**
1. Model loads successfully
2. GRPO training runs to planned stopping point without environment failure
3. Adapter weights saved and reloadable
4. Non-degenerate update vector extractable
5. Update vector consumable by analysis code
6. At least one SVD diagnostic runs successfully

This notebook is **not** a benchmark run. Do not interpret reward/accuracy numbers as H1a/H1b/H2 evidence.

## 1. Environment smoke test

In [ ]:
# Install / verify dependencies
!pip install -q trl transformers accelerate peft datasets bitsandbytes
!pip show trl transformers peft | grep -E '^(Name|Version)'

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU detected — abort"

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_supported = torch.cuda.is_bf16_supported()

print(f"GPU model  : {gpu_name}")
print(f"VRAM       : {vram_gb:.1f} GB")
print(f"bf16       : {'supported' if bf16_supported else 'NOT supported — will use float16'}")
print(f"torch      : {torch.__version__}")

# Warn if unexpected hardware
if "T4" not in gpu_name and "P100" not in gpu_name and "A100" not in gpu_name:
    print(f"WARNING: unrecognized GPU '{gpu_name}' — verify VRAM and precision settings")

# Hardware metadata for run report
HW_META = {
    "gpu_model": gpu_name,
    "vram_gb": round(vram_gb, 1),
    "bf16_supported": bf16_supported,
    "torch_version": torch.__version__,
}
print("\nHW_META:", HW_META)

## 2. Model load

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
MODEL_DTYPE = torch.bfloat16 if bf16_supported else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=MODEL_DTYPE,
    device_map="auto",
)
print("Model loaded:", MODEL_ID)
print(f"Params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 3. LoRA configuration (registered)

In [ ]:
from peft import LoraConfig, get_peft_model

# Registered adapter targets (preregistration v1.3)
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Dataset — GSM8K

In [ ]:
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main")
train_data = dataset["train"]
print(f"Train size: {len(train_data)}")
print(train_data[0])

In [ ]:
import re

SYSTEM_PROMPT = (
    "You are a math reasoning assistant. "
    "Think step by step, then end your answer with: The answer is <number>."
)

def format_example(example):
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": example["question"]},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    return {"prompt": prompt, "answer": example["answer"]}

train_data = train_data.map(format_example)
print(train_data[0]["prompt"][:300])

## 5. Reward function

In [ ]:
def extract_final_number(text):
    """Extract the last number from a GSM8K answer string."""
    matches = re.findall(r"[-+]?\d+(?:\.\d+)?", text.replace(",", ""))
    return matches[-1] if matches else None

def correctness_reward(completions, answer, **kwargs):
    """Return 1.0 if final number matches ground truth, else 0.0.

    GRPO passes completions as a flat list of length (n_prompts * num_generations)
    and answer as a list of length n_prompts. Each completion must be scored
    against its own prompt's ground truth via integer division.
    """
    n_prompts = len(answer)
    assert len(completions) % n_prompts == 0, (
        f"len(completions)={len(completions)} not divisible by n_prompts={n_prompts}"
    )
    n_gen = len(completions) // n_prompts
    rewards = []
    for i, completion in enumerate(completions):
        pred = extract_final_number(completion)
        gold = extract_final_number(answer[i // n_gen])
        rewards.append(1.0 if pred is not None and pred == gold else 0.0)
    return rewards

## 6. GRPO training (pilot — short run)

In [ ]:
import time
from trl import GRPOConfig, GRPOTrainer

# Pilot config — registered LR/LoRA settings, reduced steps for pipeline validation only
grpo_config = GRPOConfig(
    output_dir="/kaggle/working/gsm8k-qwen2.5-1.5b-phase0",
    max_steps=50,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_new_tokens=256,
    learning_rate=1e-4,
    bf16=bf16_supported,
    fp16=not bf16_supported,
    logging_steps=10,
    save_steps=50,
    save_total_limit=1,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=train_data,
    reward_funcs=correctness_reward,
    tokenizer=tokenizer,
)

t0 = time.time()
trainer.train()
total_time = time.time() - t0
step_time_sec = total_time / grpo_config.max_steps

print(f"Training complete.")
print(f"Total time    : {total_time:.1f}s")
print(f"Per-step time : {step_time_sec:.2f}s  (on {HW_META['gpu_model']})")

## 7. Save adapter checkpoint

In [ ]:
ADAPTER_PATH = "/kaggle/working/gsm8k-qwen2.5-1.5b-phase0/adapter"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Adapter saved: {ADAPTER_PATH}")

# Verify reload
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=MODEL_DTYPE, device_map="auto")
loaded = PeftModel.from_pretrained(base, ADAPTER_PATH)
print("Adapter reload: OK")

## 8. Update vector extraction

In [ ]:
import numpy as np
import hashlib, json

REGISTERED_TARGETS = set(LORA_TARGET_MODULES)

def extract_registered_update_vector(peft_model):
    """
    Extract the registered ASCENT-G update object:
        concat(Delta W_A, Delta W_B)
    over all registered LoRA layers in deterministic named_modules() order.

    Returns:
        vector       : 1-D float32 ndarray
        layer_meta   : ordered list of per-layer provenance dicts
    """
    deltas = []
    layer_meta = []
    seen_targets = set()

    for name, module in peft_model.named_modules():
        if not (hasattr(module, "lora_A") and hasattr(module, "lora_B")):
            continue

        A = module.lora_A["default"].weight.detach().float().cpu().numpy()
        B = module.lora_B["default"].weight.detach().float().cpu().numpy()

        deltas.append(A.flatten())
        deltas.append(B.flatten())
        layer_meta.append({
            "name": name,
            "a_shape": list(A.shape),
            "b_shape": list(B.shape),
            "a_numel": int(A.size),
            "b_numel": int(B.size),
            "a_norm": float(np.linalg.norm(A)),
            "b_norm": float(np.linalg.norm(B)),
        })

        for target in REGISTERED_TARGETS:
            if name.endswith(target):
                seen_targets.add(target)

    if not deltas:
        raise ValueError("No LoRA layers found — check adapter config")

    missing = REGISTERED_TARGETS - seen_targets
    if missing:
        raise ValueError(f"Missing registered target modules: {missing}")

    vector = np.concatenate(deltas)
    return vector, layer_meta


update_vector, layer_meta = extract_registered_update_vector(loaded)

vector_bytes = update_vector.tobytes()
checksum = hashlib.sha256(vector_bytes).hexdigest()

print(f"Registered update vector shape : {update_vector.shape}")
print(f"Norm                          : {np.linalg.norm(update_vector):.4f}")
print(f"Non-zero ratio                : {(update_vector != 0).mean():.4f}")
print(f"SHA-256                       : {checksum}")
print(f"Layers captured               : {len(layer_meta)}")

VECTOR_PATH = "/kaggle/working/gsm8k-qwen2.5-1.5b-phase0/update_vector.npy"
np.save(VECTOR_PATH, update_vector)

PROVENANCE_PATH = VECTOR_PATH.replace(".npy", "_provenance.json")
provenance = {
    "vector_path": VECTOR_PATH,
    "object_type": "registered_concat_lora_A_B",
    "sha256": checksum,
    "shape": list(update_vector.shape),
    "norm": float(np.linalg.norm(update_vector)),
    "registered_targets": sorted(REGISTERED_TARGETS),
    "layers": layer_meta,
}
with open(PROVENANCE_PATH, "w") as f:
    json.dump(provenance, f, indent=2)

print(f"\nRegistered update vector saved : {VECTOR_PATH}")
print(f"Provenance saved               : {PROVENANCE_PATH}")

## 9. SVD analysis — pilot diagnostic only

In [ ]:
def compute_r90(singular_values):
    """Number of singular values capturing 90% of total variance."""
    total = (singular_values ** 2).sum()
    cumvar = np.cumsum(singular_values ** 2) / total
    return int(np.searchsorted(cumvar, 0.90)) + 1

# Pilot-only diagnostic on dense effective layer deltas.
# This is separate from the registered multi-task H1a/H1b analysis,
# which operates on normalized concat(Delta W_A, Delta W_B) task vectors.
svd_deltas = []
for name, module in loaded.named_modules():
    if hasattr(module, "lora_A") and hasattr(module, "lora_B"):
        A = module.lora_A["default"].weight
        B = module.lora_B["default"].weight
        scaling = module.scaling["default"]
        delta = (scaling * (B @ A)).detach().float().cpu().numpy()
        svd_deltas.append((name, delta))

results = []
for name, delta in svd_deltas:
    U, S, Vh = np.linalg.svd(delta, full_matrices=False)
    r90 = compute_r90(S)
    results.append({"layer": name, "rank": delta.shape[1], "r90": r90, "s_max": float(S[0])})
    print(f"{name}: r90={r90}, s_max={S[0]:.4f}")

print("\nPilot SVD diagnostic: OK")

## 10. Run report

In [ ]:
import json, datetime

report = {
    "date": datetime.datetime.utcnow().isoformat(),
    "phase": 0,
    "model": MODEL_ID,
    "task": "GSM8K",
    "method": "GRPO",
    "hardware": HW_META,
    "training": {
        "max_steps": grpo_config.max_steps,
        "total_time_sec": round(total_time, 1),
        "per_step_time_sec": round(step_time_sec, 2),
        "precision": "bf16" if bf16_supported else "fp16",
        "phase0_pilot_override": {
            "max_steps": grpo_config.max_steps,
        },
    },
    "update_vector": {
        "object_type": "registered_concat_lora_A_B",
        "shape": list(update_vector.shape),
        "norm": float(np.linalg.norm(update_vector)),
        "sha256": checksum,
        "provenance_path": PROVENANCE_PATH,
        "layers_captured": len(layer_meta),
        "registered_targets": sorted(REGISTERED_TARGETS),
    },
    "svd_results": results,
    "analysis_scope": {
        "registered_primary_analysis_run": False,
        "pilot_svd_diagnostic_run": True,
    },
    "acceptance_criteria": {
        "model_loaded": True,
        "training_completed": True,
        "adapter_saved_and_reloaded": True,
        "update_vector_extracted": True,
        "update_vector_non_degenerate": bool(np.linalg.norm(update_vector) > 0),
        "registered_targets_all_covered": len(REGISTERED_TARGETS - {m["name"].split(".")[-1] for m in layer_meta}) == 0,
        "svd_diagnostic_ran": True,
    },
    "notes": "Fill in: what worked, what failed, what to fix next.",
}

REPORT_PATH = "/kaggle/working/gsm8k-qwen2.5-1.5b-phase0/run_report.json"
with open(REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print("\nPhase 0 pilot complete.")